# End-to-End BYOC Gemini Layout & Base64 BlobAssets

This notebook takes a PDF, renders its pages into full images natively with PyMuPDF, and passes the entire image to Gemini 1.5 Pro to request structural JSON spatial coordinates for charts and graphs. The script then slices the image using those bounding boxes, base64 encodes them into the `blobAssets` array, redacts them from the PyMuPDF document, and outputs the multimodal chunks.

In [ ]:
!pip install google-cloud-storage google-cloud-discoveryengine vertexai pymupdf google-auth -q

In [ ]:
import os
import time
import json
import fitz  # PyMuPDF
import base64
from google.cloud import storage
from google.cloud import discoveryengine_v1alpha as discoveryengine
from google.api_core.client_options import ClientOptions
import vertexai
from vertexai.generative_models import GenerativeModel, Part
import google.auth

# --- CONFIGURATION ---
PROJECT_ID = "YOUR_PROJECT_ID_HERE"
LOCATION = "global"

TIMESTAMP = int(time.time())
BUCKET_NAME = f"{PROJECT_ID}-byoc-staging-{TIMESTAMP}"
STAGING_DIR = "./staging"
JSONL_OUTPUT_FILE = "byoc_chunks.jsonl"

DATA_STORE_ID = f"byoc-filtered-ds-{TIMESTAMP}"
ENGINE_ID = f"byoc-filtered-app-{TIMESTAMP}"

print(f"Project ID: {PROJECT_ID}")
print(f"Bucket: {BUCKET_NAME}")
print(f"Output JSONL: {JSONL_OUTPUT_FILE}")

In [ ]:
def setup_staging_and_upload():
    if not os.path.exists(STAGING_DIR):
        os.makedirs(STAGING_DIR)
        print(f"Created staging directory at {STAGING_DIR}. Please place your PDFs here before proceeding.")
        return False

    pdf_files = [f for f in os.listdir(STAGING_DIR) if f.endswith('.pdf')]
    if not pdf_files:
        print(f"No PDFs found in {STAGING_DIR}. Please add PDFs before continuing.")
        return False

    storage_client = storage.Client(project=PROJECT_ID)
    try:
        bucket = storage_client.get_bucket(BUCKET_NAME)
        print(f"Bucket {BUCKET_NAME} already exists.")
    except Exception:
        bucket = storage_client.create_bucket(BUCKET_NAME, project=PROJECT_ID, location="US")
        print(f"Created bucket {BUCKET_NAME}")

    for filename in pdf_files:
        local_path = os.path.join(STAGING_DIR, filename)
        blob = bucket.blob(f"pdfs/{filename}")
        blob.upload_from_filename(local_path)
        print(f"Uploaded {filename} to gs://{BUCKET_NAME}/pdfs/{filename}")
    return True

ready_to_proceed = setup_staging_and_upload()

In [ ]:
if ready_to_proceed:
    vertexai.init(project=PROJECT_ID, location="us-central1")
    gemini_model = GenerativeModel("gemini-2.5-pro")
    
    from google.cloud import documentai
    
    # Initialize Document AI client
    docai_client = documentai.DocumentProcessorServiceClient(
        client_options={"api_endpoint": "us-documentai.googleapis.com"}
    )
    parent = docai_client.common_location_path(PROJECT_ID, "us")
    
    # Find or create a LAYOUT_PARSER_PROCESSOR
    processors = docai_client.list_processors(parent=parent)
    layout_processor = None
    for p in processors:
        if p.type_ == "LAYOUT_PARSER_PROCESSOR":
            layout_processor = p
            break
            
    if not layout_processor:
        print("Creating a Document AI Layout Parser Processor...")
        layout_processor = docai_client.create_processor(
            parent=parent,
            processor=documentai.Processor(
                display_name="BYOC Layout Parser",
                type_="LAYOUT_PARSER_PROCESSOR"
            )
        )
    print(f"Using Processor: {layout_processor.name}")

def describe_image_with_gemini(image_bytes):
    prompt = "Is this an image, chart, graph, or infographic? If yes, describe it in detail. If it is just text or a blank space, reply strictly with "TEXT"."
    image_part = Part.from_data(data=image_bytes, mime_type="image/png")
    try:
        response = gemini_model.generate_content([prompt, image_part])
        return response.text.strip()
    except Exception as e:
        print(f"Gemini Error: {e}")
        return "Image asset"

def extract_images_from_pdf(local_pdf_path):
    # Process the entire PDF with Document AI to get accurate layout
    with open(local_pdf_path, "rb") as f:
        pdf_content = f.read()
        
    raw_document = documentai.RawDocument(content=pdf_content, mime_type="application/pdf")
    request = documentai.ProcessRequest(name=layout_processor.name, raw_document=raw_document)
    
    print(f"Parsing layout with Document AI for {local_pdf_path}...")
    docai_result = docai_client.process_document(request=request)
    doc_data = docai_result.document
    
    pdf_doc = fitz.open(local_pdf_path)
    custom_chunks = []
    blob_assets = []
    
    for page_num, page in enumerate(pdf_doc):
        page_width = page.rect.width
        page_height = page.rect.height
        docai_page = doc_data.pages[page_num]
        
        # In Layout Parser, blocks have type 'image' 
        # Wait, docai_page.blocks may not natively expose block type easily in the client library if it's unstructured.
        # But 'visual_elements' or checking 'layout.text_anchor' helps. Actually, layout parser places images in `page.blocks`
        # and we can check if it's an image block. If layout parser fails to give 'image' type, we can fallback.
        
        # Actually Document AI Layout Parser returns images as blocks with empty text_anchor usually or 'image' type.
        # Let's iterate over blocks
        img_idx = 0
        for block in docai_page.blocks:
            # Check if block is an image or if there's an image block identifier
            # Document AI block type isn't always directly accessible. Let's assume we extract visually large blocks that aren't pure text
            # Or better, we can iterate over docai_page.visual_elements if layout parser exposes them!
            pass
            
        # Better yet, PyMuPDF has page.get_images() which gets raw images effortlessly, 
        # but the prompt asked for Doc AI to get charts/graphs/infographics.
        # Layout parser populates `page.blocks`.
        
        for block in docai_page.blocks:
            # We assume it's an image block if we see specific properties, but we'll try to get all images.
            # PyMuPDF extraction using bounding box:
            vertices = block.layout.bounding_poly.normalized_vertices
            if not vertices: continue
            
            xs = [v.x for v in vertices]
            ys = [v.y for v in vertices]
            xmin, xmax = min(xs) * page_width, max(xs) * page_width
            ymin, ymax = min(ys) * page_height, max(ys) * page_height
            
            # Simple heuristic: if it's larger than a tiny icon and has no text, or is marked as image
            # Since Document AI layout parser type isn't reliably typed in Python objects sometimes, we just crop everything
            # Wait, no. We can use page.get_images(full=True) to find raw images, but Document AI finds composite charts.
            
            # Let's try to check the block type or rely on gemini to filter.
            # If we crop and send to Gemini, we can ask Gemini "Is this an image, chart, or graph? If yes, describe. If it is just text, reply 'TEXT'."
            pass
            
        # ACTUALLY, Document AI does expose `type_` for blocks in some parsers, but Layout Parser puts images in `docai_page.blocks` where block type is not set, or it puts them in `docai_page.image_quality_scores`.
        # Wait, the documentation says Layout Parser populates blocks and gives them types like 'figure', 'table', 'text'.
        # No, let's use gemini to filter the crops. 
        
        for block in docai_page.blocks:
            # Actually, `block` doesn't have a `type` property in the protobuf for standard blocks, but it might in Document AI V1.
            # Let's just crop and pass to Gemini, but limit to blocks that look like figures.
            vertices = block.layout.bounding_poly.normalized_vertices
            if not vertices: continue
            xs = [v.x for v in vertices]
            ys = [v.y for v in vertices]
            xmin, xmax = min(xs) * page_width, max(xs) * page_width
            ymin, ymax = min(ys) * page_height, max(ys) * page_height
            
            # Skip very small blocks
            if (xmax - xmin) < 50 or (ymax - ymin) < 50:
                continue
                
            rect = fitz.Rect(xmin, ymin, xmax, ymax)
            try:
                crop_pix = page.get_pixmap(clip=rect)
                crop_bytes = crop_pix.tobytes("png")
                
                # Use Gemini to check and describe
                description = describe_image_with_gemini(crop_bytes)
                if description.upper().startswith("TEXT") or description.upper().startswith("NOT IMAGE"):
                    continue # It's just a text block
                    
                b64_image = base64.b64encode(crop_bytes).decode('utf-8')
                blob_id = f"blob_{os.path.basename(local_pdf_path)}_{page_num+1}_{img_idx}"
                
                blob_assets.append({
                    "name": blob_id,
                    "content": b64_image,
                    "mimeType": "image/png"
                })
                
                chunk = {
                    "chunkId": f"{os.path.basename(local_pdf_path)}_page{page_num+1}_img{img_idx}",
                    "content": f"Image from page {page_num+1}. Description: {description}",
                    "chunkFields": [
                        {
                            "name": "visual_asset",
                            "imageChunkField": {
                                "blobAssetId": blob_id
                            }
                        }
                    ]
                }
                custom_chunks.append(chunk)
                print(f"Page {page_num+1}, Graphic {img_idx}: Cropped and stored.")
                img_idx += 1
                
                page.add_redact_annot(rect, fill=(1,1,1))
            except Exception as e:
                print(f"Error extracting graphic rect on page {page_num+1}: {e}")
                
        page.apply_redactions()

    cleaned_path = local_pdf_path.replace(".pdf", "_cleaned.pdf")
    pdf_doc.save(cleaned_path)
    
    full_text = ""
    for page in pdf_doc:
        full_text += page.get_text() + "\n"
        
    pdf_doc.close()
    return full_text, custom_chunks, blob_assets


In [ ]:
all_jsonl_records = []

if ready_to_proceed:
    pdf_files = [f for f in os.listdir(STAGING_DIR) if f.endswith('.pdf') and not f.endswith('_cleaned.pdf')]
    
    for filename in pdf_files:
        print(f"\nProcessing {filename}...")
        local_path = os.path.join(STAGING_DIR, filename)
        
        full_text, chunks_for_doc, blob_assets = extract_images_from_pdf(local_path)
        
        if full_text.strip():
             chunks_for_doc.append({
                 "chunkId": f"{filename}_main_text",
                 "content": full_text[:8000] # Break this apart in a real prod scenario if larger
             })

        document_payload = {
             "id": filename.replace(".pdf", "").replace(".", "_").replace(" ", "_"),
             "jsonData": json.dumps({
                 "blobAssets": blob_assets,
                 "chunkedDocument": {
                     "chunks": chunks_for_doc
                 }
             }),
             "content": {
                 "mimeType": "text/plain",
                 "rawBytes": "RGVtbw=="
             }
        }
        all_jsonl_records.append(document_payload)
        
    with open(JSONL_OUTPUT_FILE, 'w') as f:
        for record in all_jsonl_records:
            f.write(json.dumps(record) + '\n')
            
    print(f"\nWrote {len(all_jsonl_records)} documents to {JSONL_OUTPUT_FILE}")
    
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.get_bucket(BUCKET_NAME)
    blob = bucket.blob(JSONL_OUTPUT_FILE)
    blob.upload_from_filename(JSONL_OUTPUT_FILE)
    print(f"Uploaded JSONL to gs://{BUCKET_NAME}/{JSONL_OUTPUT_FILE}")

### Step 5: Vertex AI Search - Create App & Ingest Data
Create an Unstructured Data Store and a Search Engine to receive the BYOC JSONL.

In [ ]:
def create_data_store():
    client_options = (
        ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com")
        if LOCATION != "global"
        else None
    )
    ds_client = discoveryengine.DataStoreServiceClient(client_options=client_options)
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"
    data_store = discoveryengine.DataStore(
        display_name="BYOC Gemini Descriptions Store",
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
    )
    print(f"Creating Data Store: {DATA_STORE_ID}...")
    try:
        operation = ds_client.create_data_store(
            request=discoveryengine.CreateDataStoreRequest(
                parent=parent, data_store_id=DATA_STORE_ID, data_store=data_store,
            )
        )
        return operation.result().name
    except Exception as e:
        print(f"Store exists or error: {e}")
        return f"{parent}/dataStores/{DATA_STORE_ID}"

def create_engine():
    client_options = (
        ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com")
        if LOCATION != "global"
        else None
    )
    engine_client = discoveryengine.EngineServiceClient(client_options=client_options)
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"
    engine = discoveryengine.Engine(
        display_name="BYOC Gemini App",
        solution_type=discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH,
        data_store_ids=[DATA_STORE_ID],
        search_engine_config=discoveryengine.Engine.SearchEngineConfig(
            search_tier=discoveryengine.SearchTier.SEARCH_TIER_ENTERPRISE,
            search_add_ons=[discoveryengine.SearchAddOn.SEARCH_ADD_ON_LLM],
        ),
    )
    print(f"Creating Engine: {ENGINE_ID}...")
    try:
        operation = engine_client.create_engine(
            request=discoveryengine.CreateEngineRequest(
                parent=parent, engine_id=ENGINE_ID, engine=engine,
            )
        )
        return operation.result().name
    except Exception as e:
        print(f"Engine exists or error: {e}")
        return f"{parent}/engines/{ENGINE_ID}"

if ready_to_proceed:
    ds_name = create_data_store()
    engine_name = create_engine()
    print("Waiting 15 seconds for backend to sync...")
    time.sleep(15)

### Step 6: Import Documents from JSONL

In [ ]:
import requests
import sys

def import_jsonl_via_rest():
    creds, project = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    access_token = creds.token

    base_url = f"https://{LOCATION}-discoveryengine.googleapis.com" if LOCATION != "global" else "https://discoveryengine.googleapis.com"
    url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/dataStores/{DATA_STORE_ID}/branches/default_branch/documents:import"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    payload = {
      "gcsSource": {
        "inputUris": [ f"gs://{BUCKET_NAME}/{JSONL_OUTPUT_FILE}" ],
        "dataSchema": "document"
      },
      "reconciliationMode": "INCREMENTAL"
    }
    print(f"Triggering Import from GCS: gs://{BUCKET_NAME}/{JSONL_OUTPUT_FILE}")
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        operation_name = response.json().get("name")
        print(f"\n[SUCCESS] Import operation initiated successfully!")
        print(f"Operation ID: {operation_name}")
        print("\n[STATUS] Search engine ingestion and embedding generation has started.")
        print("Batch processing custom chunks typically takes 10 to 15 minutes to complete.")
        print("Polling background status every 60 seconds (do not refresh)...\n")
        
        while True:
            op_url = f"{base_url}/v1alpha/{operation_name}"
            op_resp = requests.get(op_url, headers=headers)
            if op_resp.status_code == 200:
                op_data = op_resp.json()
                if op_data.get("done"):
                    error = op_data.get("error")
                    if error:
                        print(f"\n[ERROR] Import completed with errors: {error}")
                    else:
                        print(f"\n\n[SUCCESS] Document ingestion completely finished! Your Data Store is ready.")
                    break
                else:
                    sys.stdout.write("...")
                    sys.stdout.flush()
                    time.sleep(60)
            else:
                print(f"\n[ERROR] Failed to check status: {op_resp.text}")
                break
    else:
        print(f"\n[ERROR {response.status_code}]: {response.text}")

if ready_to_proceed:
    import_jsonl_via_rest()

### Step 7: Query the App using Answer API
We can now query our Vertex AI Search App to retrieve answers based on the BYOC chunks.

In [ ]:
def query_answer_rest():
    creds, project = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    access_token = creds.token

    base_url = f"https://{LOCATION}-discoveryengine.googleapis.com" if LOCATION != "global" else "https://discoveryengine.googleapis.com"
    url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}/servingConfigs/default_serving_config:answer"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    
    query_text = "What is the main topic of the uploaded PDFs?"
    payload = {
        "query": { "text": query_text },
        "answerGenerationSpec": {
            "includeCitations": True,
            "multimodalSpec": {"imageSource": "CORPUS_IMAGE_ONLY"},
            "modelSpec": { "modelVersion": "stable" }
        }
    }
    print(f"\nSending query: '{query_text}'...")
    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        data = response.json()
        answer = data.get("answer", {})
        citations = answer.get("citations", [])
        
        print("\n--- Answer ---")
        print(answer.get("answerText", "No text returned."))
        
        for citation in citations:
            for source in citation.get("sources", []):
                chunk_info = source.get("chunkInfo", {})
                blob_attachments = chunk_info.get("blobAttachments", [])
                if blob_attachments:
                    print(f"\n[SUCCESS] Found {len(blob_attachments)} Blob Attachment(s) in citation!")
                    for blob in blob_attachments:
                        print(f" -> Blob Name: {blob.get('name')}")
    else:
        print(f"\n[ERROR {response.status_code}]: {response.text}")

if ready_to_proceed:
    print("Executing final Answer query against the fully updated App.")
    query_answer_rest()

### Step 8: Clean Up Resources
Run the following cell to delete the GCS bucket, Vertex Search Engine, and Data Store generated during this notebook's execution.

In [ ]:
def clean_up_resources():
    print("Starting cleanup of resources...")
    
    # 1. Delete GCS Bucket
    try:
        storage_client = storage.Client(project=PROJECT_ID)
        bucket = storage_client.get_bucket(BUCKET_NAME)
        blobs = bucket.list_blobs()
        for blob in blobs:
            blob.delete()
        bucket.delete()
        print(f"[SUCCESS] Deleted bucket {BUCKET_NAME} and its contents.")
    except Exception as e:
        print(f"[ERROR] Failed to delete bucket {BUCKET_NAME}: {e}")
        
    # 2. Delete Engine & Data Store via REST API
    creds, _ = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    access_token = creds.token
    headers = {"Authorization": f"Bearer {access_token}"}
    
    base_url = f"https://{LOCATION}-discoveryengine.googleapis.com" if LOCATION != "global" else "https://discoveryengine.googleapis.com"
    
    # Delete Engine
    engine_url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}"
    print(f"Deleting Engine {ENGINE_ID}...")
    resp = requests.delete(engine_url, headers=headers)
    if resp.status_code == 200:
        print(f"[SUCCESS] Engine {ENGINE_ID} deletion triggered.")
    else:
        print(f"[ERROR] Failed to delete Engine {ENGINE_ID}: {resp.text}")
        
    # Delete Data Store
    ds_url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/dataStores/{DATA_STORE_ID}"
    print(f"Deleting Data Store {DATA_STORE_ID}...")
    resp = requests.delete(ds_url, headers=headers)
    if resp.status_code == 200:
        print(f"[SUCCESS] Data Store {DATA_STORE_ID} deletion triggered.")
    else:
        print(f"[ERROR] Failed to delete Data Store {DATA_STORE_ID}: {resp.text}")

if ready_to_proceed:
    print("Uncomment the line below to permanently delete the created resources.")
    # clean_up_resources()